# MRNet Knee MRI Classification — SwinViT + Attention Pooling

Trains a Swin Transformer (Swin-Tiny) on the MRNet knee MRI dataset using the same task setup and evaluation protocol as the ResNet18 notebook. The convolutional backbone is replaced by a Swin Transformer, while the learned slice attention mechanism, two-stage training protocol, 85/15 patient split, and held-out test set remain identical. Using the same random seed ensures both models train on exactly the same patients, making the final comparison fair.

In [ ]:
!pip install redivis timm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.6/775.6 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 105.9 MB/s eta 0:00:00


## Imports

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torch.nn.functional as F
import pandas as pd
import timm
import warnings
from torchvision import transforms
from PIL import Image
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


## Download Dataset

Same download as the ResNet18 notebook. If files already exist on Drive they are overwritten to ensure consistency.

In [ ]:
import redivis

scan_base = '/content/gdrive/MyDrive/MRNET_Dataset/'
os.makedirs(scan_base, exist_ok=True)

org = redivis.organization("AIMI")
src = org.dataset("mrnet_knee_mri_s:4a2c:v1_0")

src.table("train").to_directory().download(scan_base + "train", overwrite=True)
src.table("valid").to_directory().download(scan_base + "valid", overwrite=True)

for t in ['train-acl','train-meniscus','train-abnormal',
          'valid-acl','valid-meniscus','valid-abnormal']:
    src.table(t).download(scan_base + f"{t}.csv", overwrite=True)

Please visit the URL below to authenticate with your Redivis account:
https://redivis.com/oauth/authorize?user_code=6637defc51a31e72d1a0785b80109bdd


0/3390 files:   0%|          | 0.00/7.00G [00:00<?, ?B/s]

0/360 files:   0%|          | 0.00/741M [00:00<?, ?B/s]

  0%|          | 0.00/100 [00:00<?, ?%/s]

0/1 files:   0%|          | 0.00/7.92k [00:00<?, ?B/s]

...

0/1 files:   0%|          | 0.00/846 [00:00<?, ?B/s]

  0%|          | 0.00/100 [00:00<?, ?%/s]

0/1 files:   0%|          | 0.00/846 [00:00<?, ?B/s]

  0%|          | 0.00/100 [00:00<?, ?%/s]

0/1 files:   0%|          | 0.00/846 [00:00<?, ?B/s]

## Model Architecture

The backbone is Swin-Tiny pretrained on ImageNet. Unlike convolutional networks, Swin Transformer partitions each MRI slice into non-overlapping local windows and computes self-attention within each window. A shifted window mechanism in alternating layers allows information to flow across window boundaries, giving the model both fine-grained local detail and broader spatial context within each slice. The backbone outputs a 768-dimensional feature vector per slice.

The same slice attention pooling from the ResNet18 model is applied on top: a learned weighted sum aggregates slice features into a single exam-level representation, which is then classified by a linear head. The two-stage training protocol (freeze backbone for ten epochs, then fine-tune end-to-end) is also the same.

In [ ]:
class ViTKnee(nn.Module):
    def __init__(self, drop_rate=0.5):
        super().__init__()
        self.trunk    = timm.create_model(
            'swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)
        self.emb_dim  = self.trunk.num_features
        self.gate     = nn.Sequential(
            nn.Linear(self.emb_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        self.fc = nn.Sequential(
            nn.Dropout(p=drop_rate),
            nn.Linear(self.emb_dim, 1)
        )

    def forward(self, x):
        x = torch.squeeze(x, dim=0)
        if x.shape[-1] != 224:
            x = F.interpolate(x, size=(224, 224),
                              mode='bilinear', align_corners=False)
        x   = self.trunk(x)
        w   = torch.softmax(self.gate(x), dim=0)
        out = torch.sum(x * w, dim=0, keepdim=True)
        return self.fc(out).squeeze(1)

## Dataset

`SplitDataset` and `HeldOutSet` are identical to those in the ResNet18 notebook. The same seed=42 split means both models see exactly the same training and validation patients throughout all experiments.

In [ ]:
class SplitDataset(data.Dataset):
    def __init__(self, root, condition, view, patient_ids, transform=None):
        super().__init__()
        self.transform = transform
        self.folder    = os.path.join(root, 'train', view)
        df             = pd.read_csv(
            os.path.join(root, f'train-{condition}.csv'),
            header=0, names=['pid', 'label'])
        df['pid']      = df['pid'].map(lambda i: '0'*(4-len(str(i)))+str(i))
        df             = df[df['pid'].isin(patient_ids)].reset_index(drop=True)
        self.paths     = [os.path.join(self.folder, p+'.npy')
                          for p in df['pid'].tolist()]
        self.labels    = df['label'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        vol = np.load(self.paths[idx])
        lbl = torch.FloatTensor([self.labels[idx]])
        if self.transform:
            frames = []
            for s in vol:
                s_n = ((s-s.min())/(s.max()-s.min()+1e-8)*255).astype(np.uint8)
                frames.append(self.transform(Image.fromarray(s_n).convert('RGB')))
            vol = torch.stack(frames, dim=0)
        else:
            vol = (vol-vol.min())/(vol.max()-vol.min()+1e-8)
            vol = np.stack((vol,)*3, axis=1)
            vol = torch.FloatTensor(vol)
        return vol, lbl


class HeldOutSet(data.Dataset):
    def __init__(self, root, condition, view):
        super().__init__()
        self.folder = os.path.join(root, 'valid', view)
        df          = pd.read_csv(
            os.path.join(root, f'valid-{condition}.csv'),
            header=0, names=['pid', 'label'])
        df['pid']   = df['pid'].map(lambda i: '0'*(4-len(str(i)))+str(i))
        self.paths  = [os.path.join(self.folder, p+'.npy')
                       for p in df['pid'].tolist()]
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        vol = np.load(self.paths[idx])
        lbl = torch.FloatTensor([self.labels[idx]])
        vol = (vol-vol.min())/(vol.max()-vol.min()+1e-8)
        vol = np.stack((vol,)*3, axis=1)
        vol = torch.FloatTensor(vol)
        return vol, lbl

## Configuration

Checkpoints are saved to a separate `swin_split/` folder to keep them distinct from the ResNet18 checkpoints. All other hyperparameters match the ResNet18 setup.

In [ ]:
scan_base   = '/content/gdrive/MyDrive/MRNET_Dataset/'
ckpt_vault  = '/content/gdrive/MyDrive/MRNET_Dataset/checkpoints/swin_split/'
os.makedirs(ckpt_vault, exist_ok=True)

n_epochs    = 50
accum_steps = 8
patience    = 10
p_drop      = 0.5
wd          = 1e-4
split_seed  = 42
val_ratio   = 0.15

exp_grid = [
    ('acl',      'sagittal'),
    ('acl',      'coronal'),
    ('acl',      'axial'),
    ('meniscus', 'sagittal'),
    ('meniscus', 'coronal'),
    ('meniscus', 'axial'),
    ('abnormal', 'sagittal'),
    ('abnormal', 'coronal'),
    ('abnormal', 'axial'),
]

vol_transform = transforms.Compose([
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.11, 0.11)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

## Training Loop

Same structure as the ResNet18 training loop. All nine experiments run sequentially with crash detection and automatic resume. The only difference from the ResNet18 loop is that `net.trunk` is frozen and unfrozen instead of `net.encoder`.

In [ ]:
resume_map = {}
for cond, vw in exp_grid:
    lp = os.path.join(ckpt_vault, f'latest_{cond}_{vw}.pt')
    bp = os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt')
    if os.path.exists(lp) and not os.path.exists(bp):
        ck = torch.load(lp, map_location='cpu', weights_only=False)
        resume_map[(cond, vw)] = ck
        print(f"Resuming {cond}/{vw} from epoch {ck['epoch']+1}")

if not resume_map:
    print("No interrupted runs found.")

ledger = {}

for cond, vw in exp_grid:

    bp = os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt')

    if os.path.exists(bp):
        ck = torch.load(bp, map_location='cpu', weights_only=False)
        ledger[f"{cond}_{vw}"] = ck['val_auc']
        print(f"Skip {cond}/{vw} (AUC {ck['val_auc']})")
        continue

    print(f"\n{'='*50}\n{cond.upper()} | {vw}\n{'='*50}")

    ref_df        = pd.read_csv(
        os.path.join(scan_base, f'train-{cond}.csv'),
        header=0, names=['pid', 'label'])
    ref_df['pid'] = ref_df['pid'].map(lambda i: '0'*(4-len(str(i)))+str(i))

    tr_ids, vl_ids = train_test_split(
        ref_df['pid'].tolist(),
        test_size=val_ratio,
        random_state=split_seed,
        stratify=ref_df['label'].tolist()
    )

    tr_set = SplitDataset(scan_base, cond, vw, tr_ids, transform=vol_transform)
    vl_set = SplitDataset(scan_base, cond, vw, vl_ids, transform=None)

    tr_lbl  = tr_set.labels
    pos_w   = torch.tensor([
        (len(tr_lbl) - sum(tr_lbl)) / max(sum(tr_lbl), 1)])
    if torch.cuda.is_available():
        pos_w = pos_w.cuda()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    feed_ldr  = data.DataLoader(tr_set, batch_size=1, shuffle=True,  num_workers=2)
    probe_ldr = data.DataLoader(vl_set, batch_size=1, shuffle=False, num_workers=2)

    net = ViTKnee(drop_rate=p_drop)
    if torch.cuda.is_available():
        net = net.cuda()

    for p in net.trunk.parameters():
        p.requires_grad = False

    opt   = optim.Adam(filter(lambda p: p.requires_grad, net.parameters()),
                       lr=1e-3, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='max', factor=0.5, patience=3)

    peak_auc  = 0
    stall_cnt = 0
    unfrozen  = False
    ep_range  = range(n_epochs)

    rck = resume_map.get((cond, vw))
    if rck is not None:
        r_ep = rck['epoch'] + 1
        net.load_state_dict(rck['model_state_dict'])
        peak_auc  = rck['val_auc']
        stall_cnt = 0
        ep_range  = range(r_ep, n_epochs)
        if r_ep >= 10:
            for p in net.trunk.parameters():
                p.requires_grad = True
            opt = optim.Adam([
                {'params': net.trunk.parameters(), 'lr': 1e-5},
                {'params': net.gate.parameters(),  'lr': 1e-4},
                {'params': net.fc.parameters(),    'lr': 1e-4},
            ], weight_decay=wd)
            sched    = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='max', factor=0.5, patience=3)
            unfrozen = True
        try:
            opt.load_state_dict(rck['optimizer_state_dict'])
        except Exception:
            pass
        tr_ids = rck['tr_ids']
        vl_ids = rck['vl_ids']

    for ep in ep_range:

        if ep == 10 and not unfrozen:
            for p in net.trunk.parameters():
                p.requires_grad = True
            opt = optim.Adam([
                {'params': net.trunk.parameters(), 'lr': 1e-5},
                {'params': net.gate.parameters(),  'lr': 1e-4},
                {'params': net.fc.parameters(),    'lr': 1e-4},
            ], weight_decay=wd)
            sched    = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode='max', factor=0.5, patience=3)
            unfrozen = True

        net.train()
        p_tr, t_tr, l_tr = [], [], []
        opt.zero_grad()

        for i, (img, lbl) in enumerate(feed_ldr):
            if torch.cuda.is_available():
                img, lbl = img.cuda(), lbl.cuda()
            out  = net(img.float()).view(-1)
            lbl  = lbl.view(-1)
            loss = loss_fn(out, lbl) / accum_steps
            loss.backward()
            if (i+1) % accum_steps == 0 or (i+1) == len(feed_ldr):
                opt.step()
                opt.zero_grad()
            l_tr.append(loss.item() * accum_steps)
            t_tr.append(int(lbl.item()))
            p_tr.append(torch.sigmoid(out).item())

        tr_loss = np.round(np.mean(l_tr), 4)
        try:
            tr_auc = np.round(roc_auc_score(t_tr, p_tr), 4)
        except Exception:
            tr_auc = 0.5

        net.eval()
        p_vl, t_vl = [], []
        vl_loss = 0.0
        with torch.no_grad():
            for img, lbl in probe_ldr:
                if torch.cuda.is_available():
                    img, lbl = img.cuda(), lbl.cuda()
                out      = net(img.float()).view(-1)
                lbl      = lbl.view(-1)
                vl_loss += loss_fn(out, lbl).item()
                t_vl.append(int(lbl.item()))
                p_vl.append(torch.sigmoid(out).item())

        vl_loss /= len(probe_ldr)
        try:
            vl_auc = np.round(roc_auc_score(t_vl, p_vl), 4)
        except Exception:
            vl_auc = 0.5

        sched.step(vl_auc)
        print(f"  ep {ep:02d} | tr_loss {tr_loss} | tr_auc {tr_auc} | "
              f"vl_loss {np.round(vl_loss,4)} | vl_auc {vl_auc}")

        torch.save({
            'epoch': ep,
            'model_state_dict': net.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'val_auc': vl_auc,
            'tr_ids': tr_ids,
            'vl_ids': vl_ids,
        }, os.path.join(ckpt_vault, f'latest_{cond}_{vw}.pt'))

        if vl_auc > peak_auc:
            peak_auc  = vl_auc
            stall_cnt = 0
            torch.save({
                'epoch': ep,
                'model_state_dict': net.state_dict(),
                'val_auc': vl_auc,
                'tr_ids': tr_ids,
                'vl_ids': vl_ids,
            }, os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt'))
            print(f"  ✓ saved (AUC {vl_auc})")
        else:
            stall_cnt += 1

        if stall_cnt == patience:
            print(f"  early stop ep {ep}")
            break

    ledger[f"{cond}_{vw}"] = peak_auc
    print(f"\n  best val AUC {cond}/{vw}: {peak_auc}")

print("\n" + "="*52)
print(f"{'Task':<12} {'Plane':<12} {'Val AUC (15% split)'}")
print("-"*40)
for cond, vw in exp_grid:
    print(f"{cond:<12} {vw:<12} {ledger.get(f'{cond}_{vw}', 'N/A')}")
print("="*52)
print(f"Split: 85% train / 15% val  |  seed={split_seed}")
print(f"Stanford valid set (120 patients) kept fully held out.")

No interrupted runs found.

ACL | sagittal
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

  ep 00 | tr_loss 1.1201 | tr_auc 0.564 | vl_loss 0.9828 | vl_auc 0.8677
  ✓ saved (AUC 0.8677)
  ep 01 | tr_loss 1.0469 | tr_auc 0.6889 | vl_loss 0.8732 | vl_auc 0.857
  ep 02 | tr_loss 1.0197 | tr_auc 0.7139 | vl_loss 0.8752 | vl_auc 0.8515
  ep 03 | tr_loss 0.9867 | tr_auc 0.7382 | vl_loss 0.8459 | vl_auc 0.8633
  ep 04 | tr_loss 0.9883 | tr_auc 0.7389 | vl_loss 0.8156 | vl_auc 0.87
  ✓ saved (AUC 0.87)
  ep 05 | tr_loss 0.994 | tr_auc 0.7357 | vl_loss 0.8131 | vl_auc 0.8647
  ep 06 | tr_loss 0.9701 | tr_auc 0.747 | vl_loss 0.8505 | vl_auc 0.8508
  ep 07 | tr_loss 0.9225 | tr_auc 0.7815 | vl_loss 0.7946 | vl_auc 0.8608
  ep 08 | tr_loss 0.9365 | tr_auc 0.7724 | vl_loss 0.7947 | vl_auc 0.8624
  ep 09 | tr_loss 0.9019 | tr_auc 0.7939 | vl_loss 0.7969 | vl_auc 0.8612
  ep 10 | tr_loss 0.9584 | tr_auc 0.7563 | vl_loss 0.7707 | vl_auc 0.8573
  ep 11 | tr_loss 0.9649 | tr_auc 0.7554 | vl_loss 0.8169 | vl_auc 0.8443
  ep 12 | tr_loss 0.9321 | tr_auc 0.7789 | vl_loss 0.8022 | vl_auc 0.8396


## Convergence Diagnostic

Checks all nine checkpoints to verify each run completed properly. A gap of 10 between best epoch and last epoch means early stopping fired correctly. A smaller gap indicates a runtime crash and those experiments need to be resumed before running the test evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

ckpt_vault = '/content/gdrive/MyDrive/MRNET_Dataset/checkpoints/swin_split/'

exp_grid = [
    ('acl',      'sagittal'),
    ('acl',      'coronal'),
    ('acl',      'axial'),
    ('meniscus', 'sagittal'),
    ('meniscus', 'coronal'),
    ('meniscus', 'axial'),
    ('abnormal', 'sagittal'),
    ('abnormal', 'coronal'),
    ('abnormal', 'axial'),
]

print("="*70)
print(f"{'Task':<12} {'Plane':<12} {'Best ep':<10} {'Last ep':<10} {'Best AUC':<10} {'Gap':<6} {'Status'}")
print("-"*70)

issues = []
for cond, vw in exp_grid:
    bp = os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt')
    lp = os.path.join(ckpt_vault, f'latest_{cond}_{vw}.pt')
    if os.path.exists(bp) and os.path.exists(lp):
        bk      = torch.load(bp, map_location='cpu', weights_only=False)
        lk      = torch.load(lp, map_location='cpu', weights_only=False)
        best_ep = bk.get('epoch', 0)
        last_ep = lk.get('epoch', 0)
        auc     = bk.get('val_auc', 0)
        gap     = last_ep - best_ep
        if gap >= 10:
            status = '✓ converged'
        elif gap < 5:
            status = '⚠ crashed'
            issues.append((cond, vw, gap))
        else:
            status = '~ mid-patience'
            issues.append((cond, vw, gap))
        print(f"{cond:<12} {vw:<12} {best_ep:<10} {last_ep:<10} {auc:<10} {gap:<6} {status}")
    else:
        print(f"{cond:<12} {vw:<12} {'?':<10} {'?':<10} {'?':<10} {'?':<6} ✗ missing")
        issues.append((cond, vw, -1))

print("="*70)
if issues:
    print(f"\n⚠ {len(issues)} need attention: {[(c,v) for c,v,_ in issues]}")
else:
    print("\n✓ All converged.")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Task         Plane        Best ep    Last ep    Best AUC   Gap    Status
----------------------------------------------------------------------
acl          sagittal     4          14         0.87       10     ✓ converged
acl          coronal      23         33         0.8803     10     ✓ converged
acl          axial        13         23         0.8944     10     ✓ converged
meniscus     sagittal     14         24         0.8003     10     ✓ converged
meniscus     coronal      19         29         0.8356     10     ✓ converged
meniscus     axial        15         25         0.79       10     ✓ converged
abnormal     sagittal     13         23         0.9228     10     ✓ converged
abnormal     coronal      13         23         0.9239     10     ✓ converged
abnormal     axial        12         22         0.9186     10     ✓ converged

✓ All converged.


## Test Evaluation and Multi-Plane Fusion

Evaluates all nine SwinViT checkpoints on the 120 held-out Stanford patients, then runs multi-plane fusion using the same three strategies as the ResNet18 notebook. The final table compares SwinViT fusion directly against ResNet18 fusion and the original MRNet paper baseline on the same test set.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

from scipy.optimize import minimize

scan_base  = '/content/gdrive/MyDrive/MRNET_Dataset/'
ckpt_vault = '/content/gdrive/MyDrive/MRNET_Dataset/checkpoints/swin_split/'

exp_grid = [
    ('acl',      'sagittal'),
    ('acl',      'coronal'),
    ('acl',      'axial'),
    ('meniscus', 'sagittal'),
    ('meniscus', 'coronal'),
    ('meniscus', 'axial'),
    ('abnormal', 'sagittal'),
    ('abnormal', 'coronal'),
    ('abnormal', 'axial'),
]


def best_threshold(tgts, probs):
    bt, bf = 0.5, 0.0
    for t in np.arange(0.1, 0.9, 0.01):
        f = f1_score(tgts, (np.array(probs) >= t).astype(int), zero_division=0)
        if f > bf:
            bf, bt = f, t
    return bt


def eval_on_test(cond, vw):
    bp = os.path.join(ckpt_vault, f'best_{cond}_{vw}.pt')
    if not os.path.exists(bp):
        print(f"  No checkpoint: {cond}/{vw}")
        return None
    ck  = torch.load(bp, map_location='cpu', weights_only=False)
    net = ViTKnee(drop_rate=0.0)
    net.load_state_dict(ck['model_state_dict'])
    net.eval()
    if torch.cuda.is_available():
        net = net.cuda()

    ts_set = HeldOutSet(scan_base, cond, vw)
    ts_ldr = data.DataLoader(ts_set, batch_size=1, shuffle=False, num_workers=2)

    probs, tgts = [], []
    with torch.no_grad():
        for img, lbl in ts_ldr:
            if torch.cuda.is_available():
                img = img.cuda()
            prob = torch.sigmoid(net(img.float()).view(-1)).item()
            probs.append(prob)
            tgts.append(int(lbl.item()))

    t     = best_threshold(tgts, probs)
    preds = (np.array(probs) >= t).astype(int)
    auc   = np.round(roc_auc_score(tgts, probs), 4)
    f1    = np.round(f1_score(tgts, preds,        zero_division=0), 4)
    prec  = np.round(precision_score(tgts, preds,  zero_division=0), 4)
    rec   = np.round(recall_score(tgts, preds,     zero_division=0), 4)
    cm    = confusion_matrix(tgts, preds)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)
    spec  = np.round(tn/(tn+fp) if (tn+fp) > 0 else 0, 4)
    npv   = np.round(tn/(tn+fn) if (tn+fn) > 0 else 0, 4)

    return {
        'epoch': ck['epoch'], 'thresh': np.round(t, 2),
        'auc': auc, 'f1': f1, 'prec': prec,
        'sens': rec, 'spec': spec, 'npv': npv,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'val_auc': ck['val_auc'], 'probs': probs, 'tgts': tgts,
    }


print("="*92)
print("SWINVIT TEST EVALUATION — Stanford held-out valid set (120 patients)")
print("Model: SwinViT + Attention  |  Trained on 85% of training data")
print("="*92)
print(f"{'Task':<12} {'Plane':<12} {'Ep':<5} {'Thr':<6} {'Test AUC':<10} "
      f"{'F1':<8} {'Prec':<8} {'Sens':<8} {'Spec':<8} {'NPV'}")
print("-"*92)

results   = {}
all_probs = {c: {v: {'probs': [], 'tgts': []}
                 for v in ['sagittal','coronal','axial']}
             for c in ['acl','meniscus','abnormal']}

for cond, vw in exp_grid:
    r = eval_on_test(cond, vw)
    if r:
        results[f"{cond}_{vw}"] = r
        all_probs[cond][vw]['probs'] = r['probs']
        all_probs[cond][vw]['tgts']  = r['tgts']
        print(f"{cond:<12} {vw:<12} {r['epoch']:<5} {r['thresh']:<6} "
              f"{r['auc']:<10} {r['f1']:<8} {r['prec']:<8} "
              f"{r['sens']:<8} {r['spec']:<8} {r['npv']}")

print("="*92)

print("\nConfusion matrices (Stanford held-out test set):")
print("-"*55)
for cond, vw in exp_grid:
    k = f"{cond}_{vw}"
    if k in results:
        r = results[k]
        print(f"  {cond}/{vw:<14} TP={r['tp']}  TN={r['tn']}  "
              f"FP={r['fp']}  FN={r['fn']}")

print("\nVal AUC (15% split) vs Test AUC (Stanford held-out):")
print("-"*52)
for cond, vw in exp_grid:
    k = f"{cond}_{vw}"
    if k in results:
        r    = results[k]
        diff = np.round(r['auc'] - r['val_auc'], 4)
        sign = '+' if diff >= 0 else ''
        print(f"  {cond}/{vw:<14} val={r['val_auc']}  "
              f"test={r['auc']}  Δ={sign}{diff}")

print("\n" + "="*65)
print("MULTI-PLANE FUSION ON TEST SET")
print("="*65)
print(f"{'Strategy':<38} {'ACL':<10} {'Meniscus':<12} {'Abnormal'}")
print("-"*65)

fusion_res = {}
for cond in ['acl','meniscus','abnormal']:
    lbls = all_probs[cond]['sagittal']['tgts']
    sag  = np.array(all_probs[cond]['sagittal']['probs'])
    cor  = np.array(all_probs[cond]['coronal']['probs'])
    axi  = np.array(all_probs[cond]['axial']['probs'])

    eq   = np.round(roc_auc_score(lbls, (sag+cor+axi)/3), 4)
    best = np.round(max(roc_auc_score(lbls, sag),
                        roc_auc_score(lbls, cor),
                        roc_auc_score(lbls, axi)), 4)

    def neg_auc(w):
        w = np.clip(w, 0, 1); w = w / (w.sum() + 1e-8)
        return -roc_auc_score(lbls, w[0]*sag + w[1]*cor + w[2]*axi)

    res = minimize(neg_auc, [1/3,1/3,1/3], method='Nelder-Mead',
                   options={'maxiter': 1000})
    bw  = np.clip(res.x, 0, 1); bw = bw / bw.sum()
    lw  = np.round(roc_auc_score(lbls, bw[0]*sag+bw[1]*cor+bw[2]*axi), 4)

    fusion_res[cond] = {'eq': eq, 'best': best, 'lw': lw, 'w': bw.tolist()}

for lbl, key in [('Best single plane',          'best'),
                 ('Equal average fusion',        'eq'),
                 ('SwinViT learned fusion',      'lw')]:
    row = [fusion_res[c][key] for c in ['acl','meniscus','abnormal']]
    print(f"  {lbl:<36} {row[0]:<10} {row[1]:<12} {row[2]}")

print("-"*65)
print(f"  {'ResNet18 learned fusion (unbiased)':<36} {'?':<10} {'?':<12} {'?'}")
print(f"  {'MRNet paper (Bien et al. 2018)':<36} {'0.8120':<10} {'0.6980':<12} {'0.9370'}")
print("="*65)

print("\nSwinViT learned fusion weights (on test set):")
for cond in ['acl','meniscus','abnormal']:
    bw = fusion_res[cond]['w']
    print(f"  {cond:<12} sag={bw[0]:.2f}  cor={bw[1]:.2f}  axi={bw[2]:.2f}")